# Lambert Liu Runner

In [1]:
import utils as ut
import b_run_staging as b
import h_ll_runner as h
from i_hyper_tuning import Tuner
import c_clustering as c
import numpy as np
import polars as pl
from numba import set_num_threads, get_num_threads

set_num_threads(15)
get_num_threads()

15

### Loading in required data and changing to named tuples

In [ ]:
# Loading in required numpy arrays
static_configs = ut.load_json5('static_configs')
runtime_configs = ut.load_json5('runtime_configs')
base_config = ut.merge_configs(static_configs, runtime_configs)

train_test_dict = ut.load_json5("train_test_dict")
bin_metric_dict = ut.load_json5("bin_metric_dict")

user_counts = ut.load_data("user_counts", "df")
user_interactions = ut.load_data("user_interactions", "df")
user_mapping = ut.load_data("user_mapping", "df")

degen_mask = ut.load_data("degen_mask", "np")
interpolation_weights = ut.load_data("interpolation_weights", "np")

quadratic_interpolation = False

# Loading initial grids
u_init = ut.load_data("u_init", "np")
v_init = ut.load_data("v_init", "np")

p_init = ut.load_data("p_init", "np")
u_pos_init = ut.load_data("u_pos_init", "np")
v_pos_init = ut.load_data("v_pos_init", "np")

n_counts_init = ut.load_data("n_counts_init", "np")

u_clustering = ut.load_data("u_clustering", "np")
v_clustering = ut.load_data("v_clustering", "np")

u_pos_clustering = ut.load_data("u_pos_clustering", "np")
v_pos_clustering = ut.load_data("v_pos_clustering", "np")
p_pos_clustering = ut.load_data("p_pos_clustering", "np")


### Converting to named tuples

In [3]:
# Converting the dfs to nt of numpy arrays to be used for the final numba runner
user_interactions_nt = b.df_to_nt('user_interactions_nt', user_interactions)
user_counts_nt = b.df_to_nt('user_counts_nt', user_counts,)
output_idx_nt, model_idx_nt = (b.get_model_and_output_idx_nt())
train_test_nt_class = b.dictionary_to_named_tuple_class('train_test_nt',train_test_dict)
train_test_nt = train_test_nt_class(**train_test_dict)
bin_metric_nt = b.dictionary_to_named_tuple_class('bin_metric_nt', bin_metric_dict)(**bin_metric_dict)

### Creating tuner class for runs

In [4]:
# Creating user type groups for tuning
user_type_groups = (user_mapping.sort('user_id')['source_user_type'].to_numpy() == 'machine').astype(np.int8)

t = Tuner(u_init, v_init, p_init, u_pos_init, v_pos_init, u_clustering, v_clustering, u_pos_clustering, v_pos_clustering, p_pos_clustering, 
          n_counts_init, user_counts_nt, user_interactions_nt, interpolation_weights, bin_metric_nt, output_idx_nt, model_idx_nt, train_test_nt_class, user_type_groups)

# Validation Runs

### Weekly runner prep

In [ ]:
ablation_data_dir = f'{ut.data_dir}/ablations'
weekly_user_counts = user_counts.lazy()

weekly_u_parts = []
weekly_v_parts = []
weekly_p_parts = []

for day_idx in range(7):
    period_start = (train_test_dict['train_start'] + day_idx * bin_metric_dict['fine_bins_per_day'])

    period_end = (period_start + bin_metric_dict['fine_bins_per_day'])

    day_u, day_v, day_p = b.init_grid_hurdle(user_counts=weekly_user_counts, n_users=user_mapping.shape[0], coarse_bins_per_day=bin_metric_dict['coarse_bins_per_day'], 
                                             period_start=period_start, period_end=period_end, bin_metric_dict=bin_metric_dict)

    weekly_u_parts.append(day_u)
    weekly_v_parts.append(day_v)
    weekly_p_parts.append(day_p)

weekly_u_pos_init = np.concatenate(weekly_u_parts, axis=1)
weekly_v_pos_init = np.concatenate(weekly_v_parts, axis=1)
weekly_p_init = np.concatenate(weekly_p_parts, axis=1)

In [ ]:
ut.store_data(weekly_u_pos_init, 'weekly_u_pos_init', data_dir=ablation_data_dir)
ut.store_data(weekly_v_pos_init, 'weekly_v_pos_init', data_dir=ablation_data_dir)
ut.store_data(weekly_p_init, 'weekly_p_init', data_dir=ablation_data_dir)

In [ ]:
# Creating degen counts
weekly_degen_counts = weekly_user_counts.filter((pl.col('fine_bin_id') >= train_test_dict['train_start']
                                ) & (pl.col('fine_bin_id') < train_test_dict['burn_in_end']) & (pl.col('count') > 1))
weekly_degen_counts = weekly_degen_counts.with_columns(((pl.col('fine_bin_id') % bin_metric_dict['fine_bins_per_week']) // bin_metric_dict['fine_bins_per_coarse_bin']).alias('weekly_coarse_bin'))
weekly_degen_counts = weekly_degen_counts.group_by(['user_id', 'weekly_coarse_bin']).agg(pl.len().alias('n_bins')).collect(engine='streaming')
weekly_degen_mask = np.ones((user_mapping.shape[0], bin_metric_dict['coarse_bins_per_day'] * 7), dtype='bool')
non_degen_rows = weekly_degen_counts['n_bins'].to_numpy() >= static_configs['degen_threshold']
weekly_entries = weekly_degen_counts.select(['user_id', 'weekly_coarse_bin']).to_numpy()
weekly_degen_mask[weekly_entries[non_degen_rows, 0], weekly_entries[non_degen_rows, 1]] = False

ut.store_data(weekly_degen_mask, 'weekly_degen_mask', data_dir=ablation_data_dir)

#### Weekly runner

In [ ]:
experiment_name = 'ablation_weekly'
hurdle_model = True
hyperparams = ut.load_json5('hyper_choices')

# Changing inits to use weekly thing instead of daily things
weekly_u_pos_init = ut.load_data('weekly_u_pos_init', 'np', data_dir=f'{ut.data_dir}/ablations')
weekly_v_pos_init = ut.load_data('weekly_v_pos_init', 'np', data_dir=f'{ut.data_dir}/ablations')
weekly_p_init = ut.load_data('weekly_p_init', 'np', data_dir=f'{ut.data_dir}/ablations')
weekly_degen_mask = ut.load_data('weekly_degen_mask', 'np', data_dir=f'{ut.data_dir}/ablations')
weekly_n_counts_init = np.zeros_like(weekly_u_pos_init)
weekly_bin_metric_dict = bin_metric_dict.copy()

# Making the runner use a day length of a week
weekly_bin_metric_dict['fine_bins_per_day'] = bin_metric_dict['fine_bins_per_week']
weekly_bin_metric_dict['coarse_bins_per_day'] = bin_metric_dict['coarse_bins_per_day'] * 7
weekly_bin_metric_nt = b.dictionary_to_named_tuple_class('weekly_bin_metric_nt', weekly_bin_metric_dict)(**weekly_bin_metric_dict)
weekly_train_test_dict = train_test_dict.copy()

# 1 week instead of 1 day train
weekly_train_test_dict['train_end'] = bin_metric_dict['fine_bins_per_week']
weekly_train_test_dict['burn_in_start'] = bin_metric_dict['fine_bins_per_week']

In [ ]:
weekly_t = Tuner(weekly_u_pos_init, weekly_v_pos_init, weekly_p_init, weekly_u_pos_init, weekly_v_pos_init, weekly_u_pos_init, 
                 weekly_v_pos_init, weekly_u_pos_init, weekly_v_pos_init, weekly_p_init, weekly_n_counts_init, user_counts_nt, 
                 user_interactions_nt, interpolation_weights, weekly_bin_metric_nt, output_idx_nt, model_idx_nt, train_test_nt_class, user_type_groups)

weekly_results = weekly_t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_model, hyperparams=hyperparams, 
                                      train_test_dict=weekly_train_test_dict, config_dict=base_config, degen_mask=weekly_degen_mask, run_name=experiment_name)

#### Daily runner with weekly mask

In [ ]:
hyperparams = ut.load_json5('hyper_choices')
experiment_name = 'ablation_daily_weekly_mask'
hurdle_model = True

weekly_degen_mask = ut.load_data('weekly_degen_mask', 'np', data_dir=f'{ut.data_dir}/ablations')
daily_weekly_mask_results = t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_model, hyperparams=hyperparams, train_test_dict=train_test_dict, config_dict=base_config, degen_mask=weekly_degen_mask, run_name=experiment_name)

#### Quadratic interpolation runner

In [ ]:
experiment_name = 'ablation_quadratic'
hurdle_model = True
hyperparams = ut.load_json5('hyper_choices')
quadratic_interpolation_weights = ut.load_data('quadratic_interpolation_weights', 'np', data_dir=f'{ut.data_dir}/ablations')
quad_t = Tuner(u_init, v_init, p_init, u_pos_init, v_pos_init, u_clustering, v_clustering, u_pos_clustering, v_pos_clustering, p_pos_clustering, 
          n_counts_init, user_counts_nt, user_interactions_nt, quadratic_interpolation_weights, bin_metric_nt, output_idx_nt, model_idx_nt, train_test_nt_class, user_type_groups)
results = quad_t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_model,  hyperparams=hyperparams, 
                    train_test_dict=train_test_dict, config_dict=base_config, degen_mask=degen_mask, run_name=experiment_name)

#### NB runner

In [ ]:
hyperparams = ut.load_json5('hyper_choices')
experiment_name = 'ablation_nb'
hurdle_model = False
results = t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_model,  hyperparams=hyperparams, 
                    train_test_dict=train_test_dict, config_dict=base_config, degen_mask=degen_mask, run_name=experiment_name)

finished_config 1/9 in 60.4s
finished_config 2/9 in 46.4s
finished_config 3/9 in 48.0s
finished_config 4/9 in 49.7s
finished_config 5/9 in 51.3s
finished_config 6/9 in 52.3s
finished_config 7/9 in 52.4s
finished_config 8/9 in 52.3s
finished_config 9/9 in 52.9s
finished_config 1/9 in 57.9s
finished_config 2/9 in 46.6s
finished_config 3/9 in 46.6s
finished_config 4/9 in 46.6s
finished_config 5/9 in 46.3s
finished_config 6/9 in 46.3s
finished_config 7/9 in 46.7s
finished_config 8/9 in 46.5s
finished_config 9/9 in 47.1s
